# 🧠 HUẤN LUYỆN WORLD MODEL (TinyViT + CfC/AR) CHO TAY ROBOT BIONIC
Notebook này mount Google Drive, tải dataset V3 (96×96) về SSD local và huấn luyện mô hình World Model sử dụng GPU trên Google Colab.

**Yêu cầu:** Thư mục `le-wm/` phải được upload đầy đủ lên Drive tại `Bionic_Hand_LWM/le-wm/` (bao gồm toàn bộ `config/`).

### Quy trình tự động hóa hoàn chỉnh:
1. **Tải dataset V3 từ Drive về SSD local của Colab** (tránh nghẽn I/O, tăng tốc độ train ~10 lần).
2. **Tự động khôi phục (Resume)** từ checkpoint cũ trên Drive nếu có.
3. **Chế độ đồng bộ chạy ngầm (Background Auto-sync)** định kỳ mỗi 10 phút lưu ngược checkpoints từ SSD local về Drive.
4. **Đồng bộ cuối kỳ** sau khi hoàn thành 100 epochs.

### Bước 1: Kết nối Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Bước 2: Chạy Huấn luyện Tự động

In [ ]:
# ================================================================
# HUẤN LUYỆN WORLD MODEL (TinyViT + CfC/AR) — CHỐNG HỌC VẸT
# Phiên bản tinh gọn: Bỏ ghi YAML thừa, Resume đúng, Auto-backup
# ================================================================

# ── 1. CÀI ĐẶT THƯ VIỆN ────────────────────────────────────────────────
import subprocess, sys

print("🔄 1. Cài đặt thư viện...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "stable-worldmodel[train]", "h5py", "hdf5plugin", "imageio", "ncps", "timm"],
    check=True
)
print("  ✓ Xong!\n")

# ── 2. IMPORT ─────────────────────────────────────────────────────────
import os, shutil, glob, threading, time

# ── 3. ĐƯỜNG DẪN ──────────────────────────────────────────────────────
# Drive (nguồn lưu trữ bền vững)
DRIVE_BASE      = "/content/drive/MyDrive/Bionic_Hand_LWM"
DRIVE_H5        = f"{DRIVE_BASE}/bionic_hand_dataset_v3_96.h5"
DRIVE_CKPT      = f"{DRIVE_BASE}/checkpoints/vit_cfc_weights.ckpt"  # đổi tên nếu dùng AR
DRIVE_BACKUP    = f"{DRIVE_BASE}/checkpoints"
DRIVE_TRAIN_DIR = f"{DRIVE_BASE}/le-wm"

# SSD Local (đọc/ghi nhanh hơn Drive ~10x, dùng khi train)
LOCAL_BASE     = "/content/dataset"
LOCAL_DATASETS = f"{LOCAL_BASE}/datasets"
LOCAL_H5       = f"{LOCAL_DATASETS}/bionic_hand_dataset_v3_96.h5"
LOCAL_CKPT_DIR = f"{LOCAL_BASE}/checkpoints"
LOCAL_CKPT     = f"{LOCAL_CKPT_DIR}/vit_cfc_weights.ckpt"  # đổi tên nếu dùng AR

# Tạo thư mục cần thiết
for d in [LOCAL_DATASETS, LOCAL_CKPT_DIR, DRIVE_BACKUP]:
    os.makedirs(d, exist_ok=True)


# ── 4. COPY DATASET TỪ DRIVE → SSD LOCAL ──────────────────────────────
print("🔄 2. Copy dataset H5 từ Drive → SSD local...")
if not os.path.exists(DRIVE_H5):
    raise FileNotFoundError(
        f"❌ Không tìm thấy dataset tại: {DRIVE_H5}\n"
         "   Kiểm tra lại đường dẫn Google Drive."
    )
shutil.copy2(DRIVE_H5, LOCAL_H5)
print(f"  ✓ OK ({os.path.getsize(LOCAL_H5) / 1024 / 1024:.1f} MB)\n")


# ── 5. KIỂM TRA CHECKPOINT ĐỂ RESUME ───────────────────────────────────
print("🔄 3. Kiểm tra checkpoint để Resume...")

# train.py tạo ckpt_path = STABLEWM_HOME/checkpoints/{output_model_name}_weights.ckpt
# Chỉ cần copy file về LOCAL_CKPT → train.py tự phát hiện và resume.
# KHÔNG truyền ckpt_path qua CLI (Hydra báo lỗi UnknownConfigKey).

if os.path.exists(DRIVE_CKPT):
    shutil.copy2(DRIVE_CKPT, LOCAL_CKPT)
    size_mb = os.path.getsize(LOCAL_CKPT) / 1024 / 1024
    print(f"  ✓ Tìm thấy checkpoint ({size_mb:.1f} MB) → Đã copy sang SSD local")
    print(f"  → Chế độ: TỰ ĐỘNG RESUME")
else:
    print("  ℹ️  Không có checkpoint cũ trên Drive.")
    print("  → Chế độ: TRAIN MỚI từ Epoch 0")
print()



# ── 6. BACKGROUND SYNC: TỰ ĐỘNG BACKUP MỖI 10 PHÚT ─────────────────────
# SaveCkptCallback (utils.py) cũng backup mỗi 10 epochs trực tiếp.
# Thread này là lớp bảo vệ thứ 2, bắt thêm mọi file .pt/.ckpt thô.

_stop_sync = threading.Event()

def _auto_sync(interval_sec: int = 600):
    """Copy tất cả .ckpt / .pt từ SSD local lên Drive mỗi interval_sec giây."""
    while not _stop_sync.wait(interval_sec):
        files = (
            glob.glob(os.path.join(LOCAL_CKPT_DIR, "**", "*.ckpt"), recursive=True) +
            glob.glob(os.path.join(LOCAL_CKPT_DIR, "**", "*.pt"),   recursive=True)
        )
        ts = time.strftime("%H:%M:%S")
        if files:
            for f in files:
                shutil.copy2(f, DRIVE_BACKUP)
            print(f"\n  💾 [Auto-sync {ts}] Backed up {len(files)} file(s) → Drive")
        else:
            print(f"\n  ℹ️  [Auto-sync {ts}] Chưa có checkpoint để sync")

_sync_thread = threading.Thread(target=_auto_sync, args=(600,), daemon=True)
_sync_thread.start()
print("  ✅ Background auto-sync đã khởi động (mỗi 10 phút)\n")


# ── 7. HUẤN LUYỆN ─────────────────────────────────────────────────────
# Config đọc trực tiếp từ le-wm/config/ trên Drive (không ghi đè).
# CLI override: model=vit_tiny_cfc (CfC) hoặc vit_tiny_ar (AR baseline)
# Train từ scratch, không fine-tune (pretrained_weights_path không set).
print("🚀 4. Bắt đầu huấn luyện...")
print("   max_epochs=100 | model=vit_tiny_cfc | data=bionic_hand_v3_96 | precision=bf16")
print(f"   Resume: tự động nếu {LOCAL_CKPT} tồn tại")
print()

MODEL = "vit_tiny_cfc"   # hoặc "vit_tiny_ar" cho baseline
OUTPUT_NAME = "vit_cfc"   # hoặc "vit_ar" tương ứng

train_cmd = (
    f"LOCAL_DATASET_DIR={LOCAL_BASE} "
    f"STABLEWM_HOME={LOCAL_BASE} "
    f"python -u train.py "
    f"model={MODEL} "
    f"data=bionic_hand_v3_96 "
    f"trainer.max_epochs=100 "
    f"trainer.accelerator=gpu "
    f"trainer.devices=1 "
    f"wandb.enabled=False "
    f"loader.num_workers=2 "
    f"loader.persistent_workers=True "
    f"loader.prefetch_factor=2 "
    f"output_model_name={OUTPUT_NAME} "
    f"subdir={OUTPUT_NAME}_exp"
)

import subprocess
os.chdir(DRIVE_TRAIN_DIR)
process = subprocess.Popen(
    train_cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()
exit_code = process.returncode

if exit_code != 0:
    print(f"\n⚠️  Huấn luyện kết thúc bất thường (exit code: {exit_code}).")
else:
    print("\n  ✅ Huấn luyện hoàn tất thành công!")


# ── 8. DƯNG BACKGROUND SYNC VÀ SYNC LẦN CUỐI ─────────────────────────
_stop_sync.set()

print("\n🔄 5. Đồng bộ checkpoint cuối cùng lên Google Drive...")
final_files = (
    glob.glob(os.path.join(LOCAL_CKPT_DIR, "**", "*.ckpt"), recursive=True) +
    glob.glob(os.path.join(LOCAL_CKPT_DIR, "**", "*.pt"),   recursive=True)
)

if final_files:
    for f in final_files:
        shutil.copy2(f, DRIVE_BACKUP)
        print(f"  ✓ {os.path.basename(f)} → Drive")
    print(f"\n🎉 HOÀN THÀNH! Đã backup {len(final_files)} file(s) lên Drive.")
else:
    print("  ⚠️ Không tìm thấy checkpoint nào. Kiểm tra lại log huấn luyện.")
